# 03 — Model Training & Hyperparameter Tuning

**Project:** AI-Based Cybersecurity Threat Detection Using Machine Learning Techniques

Phase 2 handed us clean, feature-engineered training data. This phase trains the
**four candidate algorithms** on both datasets, tunes each one with **grid search
+ stratified k-fold cross-validation**, measures their **training time** and
**inference latency**, and saves the trained models to the model registry
(`src/models/registry/`). Phase 4 will examine them on the held-out test sets.

The training logic lives in `src/models/train.py`; this notebook calls it and
shows the results step by step.

In [1]:
# --- Setup -----------------------------------------------------------------
import sys, json, time
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from src.data import loader as L
from src.data.preprocess import Preprocessor
from src.utils import config as C
from src.models import train as T

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
print("Setup complete.")

Setup complete.


## Step A — Get the PRE-SMOTE processed features

Why not just use the balanced files saved in Phase 2? Because Phase 3 puts
**SMOTE inside the cross-validation** (see the explanation in `train.py`):
GridSearchCV needs the *unbalanced* training data and re-balances each fold's
training portion on the fly. That is the statistically honest design — if we
SMOTE'd everything up front and then split into folds, synthetic rows created
from one fold could leak into another.

So we reload the raw rows and simply re-apply the **fitted Preprocessor** that
Phase 2 saved (medians, scaling, one-hot categories). No refitting — the
preprocessor was already fitted on the training split.

In [2]:
# NSL-KDD: load the official train/test splits, apply the saved preprocessor.
nsl_train = L.load_nsl_kdd("train")
nsl_test  = L.load_nsl_kdd("test")

pp_nsl = joblib.load(C.DATA_PROCESSED_DIR / "nslkdd_preprocessor.joblib")
le_nsl = joblib.load(C.DATA_PROCESSED_DIR / "nslkdd_label_encoder.joblib")

Xtr_nsl = pp_nsl.transform(nsl_train[list(C.NSL_KDD_FEATURES)])
ytr_nsl = nsl_train[C.CATEGORY_COL].reset_index(drop=True)
Xte_nsl = pp_nsl.transform(nsl_test[list(C.NSL_KDD_FEATURES)])
yte_nsl = nsl_test[C.CATEGORY_COL].reset_index(drop=True)

print("NSL-KDD  pre-SMOTE train:", Xtr_nsl.shape, "| test:", Xte_nsl.shape)
print("  classes:", list(le_nsl.classes_))
print("  class counts (train):", ytr_nsl.value_counts().to_dict())

NSL-KDD  pre-SMOTE train: (125973, 122) | test: (22544, 122)
  classes: ['DoS', 'Normal', 'Probe', 'R2L', 'U2R']
  class counts (train): {'Normal': 67343, 'DoS': 45927, 'Probe': 11656, 'R2L': 995, 'U2R': 52}


In [3]:
# CICIDS2017: reproduce Phase 2's capped sample + 80/20 split, then transform.
cic = L.load_cicids2017_capped(per_class_cap=15_000)

cic_train, cic_test = train_test_split(
    cic, test_size=0.2, stratify=cic[C.CATEGORY_COL], random_state=C.RANDOM_STATE
)
drop_cols = [C.TYPE_COL, C.CATEGORY_COL, C.IS_ATTACK_COL, C.SOURCE_COL]
Xtr_cic_raw = cic_train.drop(columns=drop_cols)
ytr_cic = cic_train[C.CATEGORY_COL].reset_index(drop=True)
Xte_cic_raw = cic_test.drop(columns=drop_cols)
yte_cic = cic_test[C.CATEGORY_COL].reset_index(drop=True)

pp_cic = joblib.load(C.DATA_PROCESSED_DIR / "cicids_preprocessor.joblib")
le_cic = joblib.load(C.DATA_PROCESSED_DIR / "cicids_label_encoder.joblib")

Xtr_cic = pp_cic.transform(Xtr_cic_raw)
Xte_cic = pp_cic.transform(Xte_cic_raw)

print("CICIDS2017  pre-SMOTE train:", Xtr_cic.shape, "| test:", Xte_cic.shape)
print("  classes:", list(le_cic.classes_))
print("  class counts (train):", ytr_cic.value_counts().to_dict())

CICIDS2017  pre-SMOTE train: (62422, 78) | test: (15606, 78)
  classes: ['Botnet', 'Brute Force', 'DDoS', 'DoS', 'Heartbleed', 'Infiltration', 'Normal', 'PortScan', 'Web Attack']
  class counts (train): {'DoS': 12000, 'PortScan': 12000, 'DDoS': 12000, 'Normal': 12000, 'Brute Force': 11068, 'Web Attack': 1744, 'Botnet': 1572, 'Infiltration': 29, 'Heartbleed': 9}


## Step B — The four algorithms in plain words

| Algorithm | How it thinks | Strengths | Weaknesses |
|---|---|---|---|
| **Logistic Regression** | Draws a straight boundary through feature-space and asks which side each row falls on; outputs a probability. | Fast to train, fast to run, tiny memory, interpretable (weights = importance). | Only sees *linear* patterns; complex non-linear attacks may slip through. |
| **Decision Tree** | A series of yes/no questions: "is `src_bytes` > 5000?" → "is `count` < 3?" … until it reaches a leaf = a class. | Readable as a rule list; handles non-linear splits; no scaling assumptions. | One tree overfits: a single unrepresentative row can bend a branch. |
| **Random Forest** | Hundreds of trees, each trained on a random subset of rows *and* features; the forest votes. | Robust (variance is averaged away), great on tabular data, hard to overfit. | Slower to train/run; not readable as rules; more memory. |
| **XGBoost** | Trees built one after another, each one focused on the mistakes the previous ones made (boosting). | State-of-the-art on tabular data; excellent with class imbalance. | Most hyperparameters to tune; slowest; easiest to overfit if learning rate is too high. |

**Why stratified k-fold cross-validation?** "K-fold" splits the data into k
folds, trains on k−1 and scores on the held-out one, rotating until every fold
has been held out — so the score is an average over k trials, not a lucky one-off.
"Stratified" means every fold keeps the *same class proportions* as the full
data. Without it, a fold might randomly contain zero U2R attacks, and the model
would never be tested on what it must detect.

**Why `f1_macro` as the score?** Accuracy lies with imbalanced data (a model can
reach 80% by never predicting an attack). f1_macro is the average F1 over
classes, so detecting the rare classes matters as much as detecting Normal.

## Step C — Train & tune NSL-KDD (4 models)

Each cell below runs the full grid search (see `src/models/train.py`), times it,
measures inference latency (ms per row), and saves the winning model + its
metadata JSON into `src/models/registry/`.

In [4]:
# ---- NSL-KDD : all four algorithms -------------------------------------
nsl_results = {}
for name in T.MODELS:
    print(f"\n=== NSL-KDD / {name} ===", flush=True)
    res = T.tune_model(name, Xtr_nsl, ytr_nsl,
                       smote_strategy=T.SMOTE_STRATEGY["nslkdd"],
                       n_jobs=4, verbose=0)
    res["latency_ms_per_row"] = T.measure_latency(res["model"], Xte_nsl)
    T.save_model("nslkdd", res, extra_meta={
        "latency_ms_per_row": res["latency_ms_per_row"],
        "n_train_rows": len(Xtr_nsl), "n_features": Xtr_nsl.shape[1],
        "classes": list(le_nsl.classes_),
    })
    nsl_results[name] = res
    print(f"  best params : {res['best_params']}")
    print(f"  CV f1_macro : {res['cv_score_mean']:.4f} +/- {res['cv_score_std']:.4f}")
    print(f"  fit time    : {res['fit_time_s']:.1f}s  |  latency {res['latency_ms_per_row']:.4f} ms/row")


=== NSL-KDD / logistic ===


  best params : {'C': 1.0}
  CV f1_macro : 0.7432 +/- 0.0055
  fit time    : 120.4s  |  latency 0.0005 ms/row

=== NSL-KDD / decision_tree ===


  best params : {'max_depth': None, 'min_samples_leaf': 5}
  CV f1_macro : 0.8907 +/- 0.0310
  fit time    : 25.2s  |  latency 0.0004 ms/row

=== NSL-KDD / random_forest ===


  best params : {'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 200}
  CV f1_macro : 0.9333 +/- 0.0215
  fit time    : 204.7s  |  latency 0.0095 ms/row

=== NSL-KDD / xgboost ===


  best params : {'learning_rate': 0.3, 'max_depth': 3, 'n_estimators': 100}
  CV f1_macro : 0.9565 +/- 0.0177
  fit time    : 181.3s  |  latency 0.0021 ms/row


In [5]:
# ---- CICIDS2017 : all four algorithms -----------------------------------
cic_results = {}
for name in T.MODELS:
    print(f"\n=== CICIDS2017 / {name} ===", flush=True)
    res = T.tune_model(name, Xtr_cic, ytr_cic,
                       smote_strategy=T.SMOTE_STRATEGY["cicids"],
                       n_jobs=4, verbose=0)
    res["latency_ms_per_row"] = T.measure_latency(res["model"], Xte_cic)
    T.save_model("cicids", res, extra_meta={
        "latency_ms_per_row": res["latency_ms_per_row"],
        "n_train_rows": len(Xtr_cic), "n_features": Xtr_cic.shape[1],
        "classes": list(le_cic.classes_),
    })
    cic_results[name] = res
    print(f"  best params : {res['best_params']}")
    print(f"  CV f1_macro : {res['cv_score_mean']:.4f} +/- {res['cv_score_std']:.4f}")
    print(f"  fit time    : {res['fit_time_s']:.1f}s  |  latency {res['latency_ms_per_row']:.4f} ms/row")


=== CICIDS2017 / logistic ===


  best params : {'C': 10.0}
  CV f1_macro : 0.8755 +/- 0.0060
  fit time    : 66.9s  |  latency 0.0003 ms/row

=== CICIDS2017 / decision_tree ===


  best params : {'max_depth': 20, 'min_samples_leaf': 1}
  CV f1_macro : 0.9908 +/- 0.0053
  fit time    : 28.0s  |  latency 0.0003 ms/row

=== CICIDS2017 / random_forest ===


  best params : {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
  CV f1_macro : 0.9765 +/- 0.0190
  fit time    : 112.9s  |  latency 0.0052 ms/row

=== CICIDS2017 / xgboost ===


  best params : {'learning_rate': 0.3, 'max_depth': 3, 'n_estimators': 200}
  CV f1_macro : 0.9848 +/- 0.0125
  fit time    : 195.1s  |  latency 0.0047 ms/row


## Step D — Registry summary

`models_metadata.json` is a single file the web API (Phase 5) will read to
advertise the available models and their scores.

In [6]:
# Collect all per-model metadata files into one summary JSON.
meta_entries = []
for p in sorted((C.MODELS_DIR).glob("*_meta.json")):
    meta_entries.append(json.loads(p.read_text(encoding="utf-8")))

summary_path = C.MODELS_DIR / "models_metadata.json"
summary_path.write_text(json.dumps(meta_entries, indent=2), encoding="utf-8")

rows = []
for m in meta_entries:
    rows.append({
        "dataset": m["dataset"], "model": m["model"],
        "CV f1_macro": m["cv_score_mean_f1_macro"],
        "fit (s)": m["fit_time_s"],
        "latency ms/row": m["latency_ms_per_row"],
    })
summary = pd.DataFrame(rows).sort_values(["dataset", "model"])
print("Saved:", summary_path)
print()
print(summary.to_string(index=False))

Saved: D:\Cyber threat Detection\src\models\registry\models_metadata.json

dataset         model  CV f1_macro  fit (s)  latency ms/row
 cicids decision_tree       0.9908    27.98          0.0003
 cicids      logistic       0.8755    66.90          0.0003
 cicids random_forest       0.9765   112.93          0.0052
 cicids       xgboost       0.9848   195.12          0.0047
 nslkdd decision_tree       0.8907    25.18          0.0004
 nslkdd      logistic       0.7432   120.41          0.0005
 nslkdd random_forest       0.9333   204.66          0.0095
 nslkdd       xgboost       0.9565   181.30          0.0021


## Summary

* All four algorithms were tuned with grid search + **stratified 5-fold CV** on
  both datasets. SMOTE was applied **inside** each fold (via an imblearn
  pipeline) to avoid leakage.
* Best hyperparameters, CV f1_macro, training time and inference latency are
  stored per model in `src/models/registry/`.
* Phase 4 holds out the untouched test sets and builds the final comparison:
  accuracy, precision, recall, F1, ROC-AUC and confusion matrices.